## 01 — Auditoría de Datos

### Objetivo

El objetivo de este notebook es validar la estructura, calidad y significado de negocio de los datos transaccionales antes de construir métricas, análisis o modelos.

La auditoría se centra en:

- entender la granularidad del dataset;
- validar las variables de cantidad, devoluciones y precio;
- identificar valores nulos, duplicados y registros inconsistentes;
- comprender la relación entre `price`, `rrp` y `voucherAmount`;
- comprobar el nivel de agregación correcto de las variables promocionales;
- definir qué registros pueden utilizarse de forma fiable en la futura capa analítica.

La finalidad de esta fase no es limpiar el dataset de forma inmediata, sino comprender primero cómo están representados los datos y documentar las reglas que deberán aplicarse posteriormente.

> **Principio seguido:** entender el dato antes de transformarlo.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
DATA_PATH = Path("../data/raw")

train_path = DATA_PATH / "orders_train.txt"
class_path = DATA_PATH / "orders_class.txt"

print(train_path)
print(class_path)

..\data\raw\orders_train.txt
..\data\raw\orders_class.txt


In [3]:
sample = pd.read_csv(
    train_path,
    sep=";",
    nrows=10
)

sample

,orderID,orderDate,articleID,colorCode,sizeCode,productGroup,quantity,price,rrp,voucherID,voucherAmount,customerID,deviceID,paymentMethod,returnQuantity
0,a1000001,2014-01-01,i1000382,1972,44,3,1,10.00,29.99,0,0,c1010575,2,BPRG,0
1,a1000001,2014-01-01,i1000550,3854,44,3,1,20.00,39.99,0,0,c1010575,2,BPRG,0
2,a1000002,2014-01-01,i1001991,2974,38,8,1,35.00,49.99,0,0,c1045905,4,BPRG,0
3,a1000002,2014-01-01,i1001999,1992,38,8,1,49.99,49.99,0,0,c1045905,4,BPRG,1
4,a1000003,2014-01-01,i1001942,1968,42,8,1,10.00,35.99,0,0,c1089295,2,PAYPALVC,0
5,a1000003,2014-01-01,i1001942,1972,42,8,1,10.00,35.99,0,0,c1089295,2,PAYPALVC,0
6,a1000003,2014-01-01,i1001974,1001,42,8,1,25.00,39.99,0,0,c1089295,2,PAYPALVC,0
7,a1000003,2014-01-01,i1001976,3976,44,8,1,15.00,39.99,0,0,c1089295,2,PAYPALVC,0
8,a1000003,2014-01-01,i1002392,2493,42,13,0,0.00,59.99,0,0,c1089295,2,PAYPALVC,0
9,a1000004,2014-01-01,i1002457,1001,42,14,1,89.99,89.99,0,0,c1050116,3,BPRG,1


In [4]:
sample.columns.tolist()

['orderID',
 'orderDate',
 'articleID',
 'colorCode',
 'sizeCode',
 'productGroup',
 'quantity',
 'price',
 'rrp',
 'voucherID',
 'voucherAmount',
 'customerID',
 'deviceID',
 'paymentMethod',
 'returnQuantity']

In [5]:
sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   orderID         10 non-null     str    
 1   orderDate       10 non-null     str    
 2   articleID       10 non-null     str    
 3   colorCode       10 non-null     int64  
 4   sizeCode        10 non-null     int64  
 5   productGroup    10 non-null     int64  
 6   quantity        10 non-null     int64  
 7   price           10 non-null     float64
 8   rrp             10 non-null     float64
 9   voucherID       10 non-null     int64  
 10  voucherAmount   10 non-null     int64  
 11  customerID      10 non-null     str    
 12  deviceID        10 non-null     int64  
 13  paymentMethod   10 non-null     str    
 14  returnQuantity  10 non-null     int64  
dtypes: float64(2), int64(8), str(5)
memory usage: 1.7 KB


In [6]:
train = pd.read_csv(
    train_path,
    sep=";",
    na_values="NA"
)

train.shape

(2325165, 15)

In [7]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 2325165 entries, 0 to 2325164
Data columns (total 15 columns):
 #   Column          Dtype  
---  ------          -----  
 0   orderID         str    
 1   orderDate       str    
 2   articleID       str    
 3   colorCode       int64  
 4   sizeCode        str    
 5   productGroup    float64
 6   quantity        int64  
 7   price           float64
 8   rrp             float64
 9   voucherID       str    
 10  voucherAmount   float64
 11  customerID      str    
 12  deviceID        int64  
 13  paymentMethod   str    
 14  returnQuantity  int64  
dtypes: float64(4), int64(4), str(7)
memory usage: 360.6 MB


In [8]:
train.isna().sum().sort_values(ascending=False)

productGroup      351
rrp               351
voucherID           6
articleID           0
colorCode           0
orderDate           0
orderID             0
quantity            0
sizeCode            0
price               0
voucherAmount       0
customerID          0
deviceID            0
paymentMethod       0
returnQuantity      0
dtype: int64

In [9]:
train["orderDate"].agg(["min", "max"])

min    2014-01-01
max    2015-09-30
Name: orderDate, dtype: str

In [10]:
train[
    [
        "quantity",
        "returnQuantity",
        "price",
        "rrp",
        "voucherAmount"
    ]
].describe()

,quantity,returnQuantity,price,rrp,voucherAmount
count,2.325165e+06,2.325165e+06,2.325165e+06,2.324814e+06,2.325165e+06
mean,9.955431e-01,5.214077e-01,3.539911e+01,4.298515e+01,2.153040e+00
std,1.640278e-01,5.036700e-01,2.266248e+01,2.401104e+01,6.537824e+00
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.000000e+00,0.000000e+00,2.000000e+01,2.999000e+01,0.000000e+00
50%,1.000000e+00,1.000000e+00,2.999000e+01,3.999000e+01,0.000000e+00
75%,1.000000e+00,1.000000e+00,4.000000e+01,4.999000e+01,0.000000e+00
max,2.400000e+01,5.000000e+00,6.237600e+02,7.999900e+02,3.000000e+02


In [11]:
(train["returnQuantity"] > train["quantity"]).sum()

np.int64(9)

In [12]:
train.loc[
    train["quantity"] > 1,
    ["quantity", "price", "rrp", "voucherAmount"]
].head(20)

,quantity,price,rrp,voucherAmount
125,2,30.00,19.99,0.0
129,2,35.98,17.99,0.0
250,2,35.98,17.99,0.0
476,2,30.00,19.99,0.0
477,2,30.00,19.99,0.0
635,3,45.00,22.99,10.0
958,2,45.98,22.99,0.0
1006,2,139.98,69.99,0.0
1056,2,35.98,17.99,0.0
1057,2,39.98,19.99,0.0


In [13]:
train.loc[
    train["quantity"] == 1,
    ["quantity", "price", "rrp", "voucherAmount"]
].head(20)

,quantity,price,rrp,voucherAmount
0,1,10.00,29.99,0.0
1,1,20.00,39.99,0.0
2,1,35.00,49.99,0.0
3,1,49.99,49.99,0.0
4,1,10.00,35.99,0.0
5,1,10.00,35.99,0.0
6,1,25.00,39.99,0.0
7,1,15.00,39.99,0.0
9,1,89.99,89.99,0.0
10,1,15.00,39.99,0.0


In [14]:
anomalies_returns = train.loc[
    train["returnQuantity"] > train["quantity"],
    [
        "orderID",
        "orderDate",
        "articleID",
        "colorCode",
        "sizeCode",
        "productGroup",
        "quantity",
        "price",
        "rrp",
        "voucherAmount",
        "customerID",
        "returnQuantity"
    ]
]

anomalies_returns

,orderID,orderDate,articleID,colorCode,sizeCode,productGroup,quantity,price,rrp,voucherAmount,customerID,returnQuantity
113855,a1037804,2014-02-11,i1002271,6829,38,13.0,0,0.00,69.99,0.0,c1094246,1
249943,a1083057,2014-03-27,i1002395,7010,44,13.0,0,0.00,69.99,10.0,c1073681,1
263101,a1087257,2014-03-31,i1002457,1001,40,14.0,0,0.00,89.99,0.0,c1120490,1
1163056,a1372183,2014-11-27,i1002141,6692,42,9.0,0,0.00,59.99,0.0,c1212962,1
1504721,a1487536,2015-03-17,i1002509,1061,40,14.0,0,0.00,69.99,0.0,c1246647,1
1684496,a1545665,2015-05-02,i1003850,1075,38,1.0,0,0.00,89.99,0.0,c1053872,1
1922089,a1618866,2015-06-23,i1003586,7159,26,2.0,0,0.00,49.99,10.0,c1005304,1
2174447,a1697077,2015-08-23,i1002183,1001,40,9.0,1,29.99,29.99,10.0,c1226184,2
2278857,a1729416,2015-09-18,i1002232,3097,42,13.0,0,0.00,69.99,0.0,c1307819,1


In [15]:
zero_quantity = train.loc[
    train["quantity"] == 0
]

zero_quantity.shape

(31643, 15)

In [16]:
zero_quantity[
    [
        "orderID",
        "orderDate",
        "articleID",
        "quantity",
        "price",
        "rrp",
        "voucherAmount",
        "returnQuantity"
    ]
].head(30)

,orderID,orderDate,articleID,quantity,price,rrp,voucherAmount,returnQuantity
8,a1000003,2014-01-01,i1002392,0,0.0,59.99,0.0,0
13,a1000006,2014-01-01,i1001957,0,0.0,39.99,0.0,0
441,a1000153,2014-01-01,i1000408,0,0.0,25.99,0.0,0
1011,a1000337,2014-01-01,i1000550,0,0.0,39.99,0.0,0
1117,a1000365,2014-01-01,i1001986,0,0.0,49.99,0.0,0
1354,a1000430,2014-01-02,i1001498,0,0.0,59.99,0.0,0
1426,a1000459,2014-01-02,i1001364,0,0.0,59.99,0.0,0
1898,a1000634,2014-01-02,i1002973,0,0.0,22.99,0.0,0
1957,a1000666,2014-01-02,i1000545,0,0.0,45.99,0.0,0
2015,a1000685,2014-01-02,i1000506,0,0.0,35.99,0.0,0


In [17]:
discount_sample = train.loc[
    (train["quantity"] > 0) &
    (train["voucherAmount"] > 0),
    [
        "quantity",
        "price",
        "rrp",
        "voucherAmount"
    ]
].copy()

discount_sample["rrp_total"] = (
    discount_sample["rrp"] * discount_sample["quantity"]
)

discount_sample["discount_vs_rrp"] = (
    discount_sample["rrp_total"] - discount_sample["price"]
)

discount_sample.head(30)

,quantity,price,rrp,voucherAmount,rrp_total,discount_vs_rrp
47,1,25.00,29.99,10.0,29.99,4.99
48,1,39.99,39.99,10.0,39.99,0.00
111,1,15.00,35.99,10.0,35.99,20.99
112,1,49.99,49.99,10.0,49.99,0.00
113,1,39.99,39.99,10.0,39.99,0.00
305,1,25.00,49.99,10.0,49.99,24.99
306,1,25.00,49.99,10.0,49.99,24.99
328,1,59.99,59.99,10.0,59.99,0.00
329,1,22.99,22.99,10.0,22.99,0.00
330,1,30.00,35.99,10.0,35.99,5.99


In [18]:
train["quantity"].value_counts().sort_index()

quantity
0       31643
1     2275335
2       16168
3        1340
4         325
5         349
7           1
9           1
11          1
15          1
24          1
Name: count, dtype: int64

In [19]:
print(anomalies_returns.to_string(index=False))

 orderID  orderDate articleID  colorCode sizeCode  productGroup  quantity  price   rrp  voucherAmount customerID  returnQuantity
a1037804 2014-02-11  i1002271       6829       38          13.0         0   0.00 69.99            0.0   c1094246               1
a1083057 2014-03-27  i1002395       7010       44          13.0         0   0.00 69.99           10.0   c1073681               1
a1087257 2014-03-31  i1002457       1001       40          14.0         0   0.00 89.99            0.0   c1120490               1
a1372183 2014-11-27  i1002141       6692       42           9.0         0   0.00 59.99            0.0   c1212962               1
a1487536 2015-03-17  i1002509       1061       40          14.0         0   0.00 69.99            0.0   c1246647               1
a1545665 2015-05-02  i1003850       1075       38           1.0         0   0.00 89.99            0.0   c1053872               1
a1618866 2015-06-23  i1003586       7159       26           2.0         0   0.00 49.99           

In [20]:
zero_quantity["returnQuantity"].value_counts().sort_index()

returnQuantity
0    31635
1        8
Name: count, dtype: int64

In [21]:
zero_quantity[
    ["price", "rrp", "voucherAmount", "returnQuantity"]
].describe()

,price,rrp,voucherAmount,returnQuantity
count,31643.0,31643.000000,31643.000000,31643.000000
mean,0.0,46.610743,2.480184,0.000253
std,0.0,25.132900,7.546912,0.015899
min,0.0,0.000000,0.000000,0.000000
25%,0.0,29.990000,0.000000,0.000000
50%,0.0,39.990000,0.000000,0.000000
75%,0.0,59.990000,0.000000,0.000000
max,0.0,299.990000,250.000000,1.000000


In [22]:
(zero_quantity["returnQuantity"] > 0).sum()

np.int64(8)

In [23]:
voucher_orders = (
    train.loc[train["voucherAmount"] > 0]
    .groupby("orderID")
    .agg(
        n_lines=("articleID", "size"),
        voucher_values=("voucherAmount", "nunique"),
        voucher_amount=("voucherAmount", "first"),
        voucher_sum_if_added=("voucherAmount", "sum"),
        order_price=("price", "sum")
    )
)

voucher_orders.head(20)

,n_lines,voucher_values,voucher_amount,voucher_sum_if_added,order_price
orderID,,,,,
a1000016,2,1,10.0,20.0,64.99
a1000042,3,1,10.0,30.0,104.98
a1000115,2,1,10.0,20.0,50.00
a1000120,5,1,10.0,50.0,198.96
a1000179,5,1,5.0,25.0,174.98
a1000190,3,1,10.0,30.0,62.00
a1000211,3,1,10.0,30.0,75.00
a1000219,5,1,10.0,50.0,171.95
a1000263,2,1,10.0,20.0,89.99


In [24]:
voucher_orders["n_lines"].describe()

count    117677.000000
mean          3.462758
std           2.516599
min           1.000000
25%           2.000000
50%           3.000000
75%           4.000000
max          39.000000
Name: n_lines, dtype: float64

In [25]:
check_price = train.loc[
    train["quantity"] > 0,
    ["quantity", "price", "rrp"]
].copy()

check_price["rrp_total"] = (
    check_price["quantity"] * check_price["rrp"]
)

check_price["price_equals_rrp_total"] = np.isclose(
    check_price["price"],
    check_price["rrp_total"],
    equal_nan=False
)

check_price["price_equals_rrp_total"].value_counts(normalize=True)

price_equals_rrp_total
True     0.65964
False    0.34036
Name: proportion, dtype: float64

In [26]:
# ¿Los pedidos con quantity=0 contienen también líneas compradas normalmente?

zero_order_ids = train.loc[
    train["quantity"] == 0,
    "orderID"
].unique()

zero_order_context = train[
    train["orderID"].isin(zero_order_ids)
]

zero_order_summary = (
    zero_order_context
    .groupby("orderID")
    .agg(
        lines=("articleID", "size"),
        total_quantity=("quantity", "sum"),
        zero_quantity_lines=("quantity", lambda x: (x == 0).sum()),
        total_returns=("returnQuantity", "sum")
    )
)

zero_order_summary.head(20)

,lines,total_quantity,zero_quantity_lines,total_returns
orderID,,,,
a1000003,5,4,1,0
a1000006,2,1,1,0
a1000153,9,8,1,3
a1000337,6,5,1,2
a1000365,4,3,1,1
a1000430,3,2,1,2
a1000459,12,11,1,8
a1000634,2,1,1,1
a1000666,5,4,1,2


In [27]:
(zero_order_summary["total_quantity"] > 0).value_counts(normalize=True)

total_quantity
True    1.0
Name: proportion, dtype: float64

In [28]:
voucher_check = (
    train.groupby("orderID")
    .agg(
        n_lines=("articleID", "size"),
        voucher_amount_values=("voucherAmount", "nunique"),
        voucher_id_values=("voucherID", "nunique"),
        voucher_amount=("voucherAmount", "first")
    )
)

voucher_check.loc[
    voucher_check["voucher_amount"] > 0,
    ["n_lines", "voucher_amount_values", "voucher_id_values", "voucher_amount"]
].head(20)

,n_lines,voucher_amount_values,voucher_id_values,voucher_amount
orderID,,,,
a1000016,2,1,1,10.0
a1000042,3,1,1,10.0
a1000115,2,1,1,10.0
a1000120,5,1,1,10.0
a1000179,5,1,1,5.0
a1000190,3,1,1,10.0
a1000211,3,1,1,10.0
a1000219,5,1,1,10.0
a1000263,2,1,1,10.0


In [29]:
voucher_check.loc[
    voucher_check["voucher_amount"] > 0,
    ["voucher_amount_values", "voucher_id_values"]
].value_counts()

voucher_amount_values  voucher_id_values
1                      1                    117676
                       0                         1
Name: count, dtype: int64

In [30]:
price_analysis = train.loc[
    (train["quantity"] > 0) &
    (train["rrp"].notna()),
    ["quantity", "price", "rrp"]
].copy()

price_analysis["rrp_total"] = (
    price_analysis["quantity"] * price_analysis["rrp"]
)

price_analysis["price_gap"] = (
    price_analysis["price"] - price_analysis["rrp_total"]
)

price_analysis["price_status"] = np.select(
    [
        np.isclose(price_analysis["price_gap"], 0),
        price_analysis["price_gap"] < 0,
        price_analysis["price_gap"] > 0
    ],
    [
        "full_price",
        "below_rrp",
        "above_rrp"
    ],
    default="unknown"
)

price_analysis["price_status"].value_counts(normalize=True)

price_status
full_price    0.659741
below_rrp     0.337743
above_rrp     0.002516
Name: proportion, dtype: float64

In [31]:
price_analysis.loc[
    price_analysis["price_status"] == "above_rrp"
].sort_values(
    "price_gap",
    ascending=False
).head(20)

,quantity,price,rrp,rrp_total,price_gap,price_status
773158,3,179.97,40.0,120.0,59.97,above_rrp
208675,3,149.97,35.0,105.0,44.97,above_rrp
749511,2,119.98,40.0,80.0,39.98,above_rrp
792721,2,119.98,40.0,80.0,39.98,above_rrp
650563,2,119.98,40.0,80.0,39.98,above_rrp
438812,2,119.98,40.0,80.0,39.98,above_rrp
446462,2,119.98,40.0,80.0,39.98,above_rrp
647195,2,119.98,40.0,80.0,39.98,above_rrp
475920,2,119.98,40.0,80.0,39.98,above_rrp
675980,2,119.98,40.0,80.0,39.98,above_rrp


In [32]:
train.duplicated().sum()

np.int64(0)

In [33]:
train.loc[
    (train["quantity"] > 0) &
    (train["price"] == 0)
].shape

(15200, 15)

In [34]:
train.isna().sum().sort_values(ascending=False)

productGroup      351
rrp               351
voucherID           6
articleID           0
colorCode           0
orderDate           0
orderID             0
quantity            0
sizeCode            0
price               0
voucherAmount       0
customerID          0
deviceID            0
paymentMethod       0
returnQuantity      0
dtype: int64

In [35]:
zero_price = train.loc[
    (train["quantity"] > 0) &
    (train["price"] == 0)
].copy()

zero_price[
    [
        "quantity",
        "rrp",
        "voucherAmount",
        "returnQuantity"
    ]
].describe()

,quantity,rrp,voucherAmount,returnQuantity
count,15200.000000,15200.0,15200.000000,15200.000000
mean,1.000263,0.0,0.001316,0.105395
std,0.016221,0.0,0.114704,0.307071
min,1.000000,0.0,0.000000,0.000000
25%,1.000000,0.0,0.000000,0.000000
50%,1.000000,0.0,0.000000,0.000000
75%,1.000000,0.0,0.000000,0.000000
max,2.000000,0.0,10.000000,1.000000


In [36]:
zero_price["voucherAmount"].value_counts().head(20)

voucherAmount
0.0     15198
10.0        2
Name: count, dtype: int64

In [37]:
zero_price["returnQuantity"].value_counts().sort_index()

returnQuantity
0    13598
1     1602
Name: count, dtype: int64

In [38]:
zero_price["articleID"].nunique()

15

In [39]:
zero_price["articleID"].value_counts().head(20)

articleID
i1003985    2847
i1003989    2427
i1003996    2171
i1003993    1624
i1003994    1333
i1003992     948
i1003998     940
i1003988     689
i1003995     648
i1003991     593
i1003987     536
i1003997     218
i1003990     160
i1003984      35
i1003986      31
Name: count, dtype: int64

In [40]:
zero_price["productGroup"].value_counts(dropna=False).head(20)

productGroup
45.0    12287
43.0     2882
90.0       31
Name: count, dtype: int64

In [41]:
zero_price_orders = train[
    train["orderID"].isin(zero_price["orderID"])
]

zero_price_order_summary = (
    zero_price_orders
    .groupby("orderID")
    .agg(
        total_lines=("articleID", "size"),
        total_quantity=("quantity", "sum"),
        order_value=("price", "sum"),
        zero_price_lines=("price", lambda x: (x == 0).sum())
    )
)

zero_price_order_summary.head(20)

,total_lines,total_quantity,order_value,zero_price_lines
orderID,,,,
a1000558,3,3,85.98,1
a1001083,2,2,49.99,1
a1001380,3,3,55.98,1
a1001886,3,3,65.99,1
a1002028,6,5,120.00,2
a1002165,2,2,49.99,1
a1002445,7,7,111.00,1
a1002833,3,3,170.00,1
a1003296,3,3,95.98,1


In [42]:
(zero_price_order_summary["order_value"] > 0).value_counts(normalize=True)

order_value
True     0.999737
False    0.000263
Name: proportion, dtype: float64

In [43]:
(
    train["rrp"].isna() ==
    train["productGroup"].isna()
).value_counts()

True    2325165
Name: count, dtype: int64

In [44]:
missing_product = train.loc[
    train["rrp"].isna() |
    train["productGroup"].isna(),
    [
        "orderID",
        "articleID",
        "quantity",
        "price",
        "rrp",
        "productGroup",
        "returnQuantity"
    ]
]

missing_product.head(20)

,orderID,articleID,quantity,price,rrp,productGroup,returnQuantity
8118,a1002753,i1004001,1,25.0,NaN,NaN,0
13518,a1004534,i1004001,1,50.0,NaN,NaN,0
18910,a1006402,i1004001,1,25.0,NaN,NaN,0
27762,a1009374,i1004001,1,50.0,NaN,NaN,0
51237,a1016897,i1004001,1,25.0,NaN,NaN,0
51238,a1016897,i1004001,1,50.0,NaN,NaN,0
60573,a1020057,i1004001,1,25.0,NaN,NaN,0
65775,a1021841,i1004001,1,25.0,NaN,NaN,0
66899,a1022201,i1004001,1,25.0,NaN,NaN,0
71934,a1023777,i1004001,1,25.0,NaN,NaN,0


In [45]:
missing_product["articleID"].nunique()

1

In [46]:
train.loc[
    train["voucherID"].isna(),
    [
        "orderID",
        "articleID",
        "quantity",
        "price",
        "rrp",
        "voucherAmount",
        "voucherID"
    ]
]

,orderID,articleID,quantity,price,rrp,voucherAmount,voucherID
51237,a1016897,i1004001,1,25.00,NaN,0.0,NaN
51238,a1016897,i1004001,1,50.00,NaN,0.0,NaN
899762,a1284611,i1004001,1,25.00,NaN,0.0,NaN
1903295,a1612957,i1004001,1,25.00,NaN,0.0,NaN
2225949,a1712702,i1002185,1,59.99,59.99,50.0,NaN
2225950,a1712702,i1002611,1,27.99,27.99,50.0,NaN


In [47]:
train.loc[
    train["voucherID"].isna(),
    [
        "orderID",
        "articleID",
        "quantity",
        "price",
        "rrp",
        "voucherAmount",
        "voucherID"
    ]
]

,orderID,articleID,quantity,price,rrp,voucherAmount,voucherID
51237,a1016897,i1004001,1,25.00,NaN,0.0,NaN
51238,a1016897,i1004001,1,50.00,NaN,0.0,NaN
899762,a1284611,i1004001,1,25.00,NaN,0.0,NaN
1903295,a1612957,i1004001,1,25.00,NaN,0.0,NaN
2225949,a1712702,i1002185,1,59.99,59.99,50.0,NaN
2225950,a1712702,i1002611,1,27.99,27.99,50.0,NaN


## Conclusiones de la auditoría

La auditoría inicial ha permitido identificar varios aspectos de calidad y semántica del dato que deben resolverse antes de construir métricas de rendimiento de producto.

### 1. Registros con `quantity = 0`

Se han identificado **31.643 registros con `quantity = 0`**.

Todos aparecen dentro de pedidos que contienen también al menos una línea con cantidad positiva.

#### Decisión

Estos registros no se considerarán demanda efectiva ni unidades vendidas en los KPIs transaccionales.

Se conservarán en el dataset original, ya que su significado operativo exacto no puede determinarse con la información disponible.

---

### 2. Inconsistencias entre unidades compradas y devueltas

Se han detectado **9 registros en los que `returnQuantity > quantity`**.

Ocho de ellos coinciden con líneas de `quantity = 0`, mientras que existe un registro con:

- `quantity = 1`
- `returnQuantity = 2`

#### Decisión

Estos registros se excluirán de la futura capa analítica utilizada para métricas de devoluciones.

No se corregirán manualmente los valores para evitar introducir información no respaldada por los datos.

---

### 3. Granularidad de `price`

El análisis muestra que `price` representa el **importe total de la línea de pedido** y no necesariamente el precio unitario.

Por tanto, no debe calcularse el valor de venta como:

`price × quantity`

ya que esto duplicaría el importe.

Cuando sea necesario, el precio unitario podrá derivarse como:

`unit_selling_price = price / quantity`

siempre que `quantity > 0`.

---

### 4. Granularidad del voucher

`voucherAmount` y `voucherID` se repiten en las diferentes líneas que pertenecen a un mismo pedido.

#### Decisión

El voucher debe tratarse a nivel de **pedido**, no de línea.

Sumar `voucherAmount` directamente sobre todas las líneas sobreestimaría el valor promocional.

---

### 5. Relación entre precio de venta y RRP

Para los registros con cantidad positiva y RRP disponible:

- **65,97%** se venden al RRP
- **33,77%** se venden por debajo del RRP
- **0,25%** se venden por encima del RRP

#### Decisión

Se crearán categorías de estado de precio:

- `full_price`
- `below_rrp`
- `above_rrp`
- `zero_value`
- `unknown_rrp`

El markdown y el uso de voucher se tratarán como conceptos comerciales distintos.

---

### 6. Artículos con valor cero

Se han identificado **15.200 registros con `quantity > 0`, `price = 0` y `rrp = 0`**.

Estos registros pertenecen únicamente a **15 artículos diferentes** y casi todos aparecen dentro de pedidos que contienen también productos de pago.

#### Decisión

Se conservarán e identificarán mediante una variable específica, pero se excluirán de métricas relacionadas con pricing, markdown y valor comercial.

No se asumirá que son regalos o productos promocionales sin evidencia adicional.

---

### 7. Metadata de producto incompleta

Se han identificado **351 registros con valores nulos tanto en `rrp` como en `productGroup`**.

Todos corresponden al mismo artículo:

`i1004001`

#### Decisión

El artículo podrá utilizarse para análisis de unidades y devoluciones, pero no para métricas que necesiten RRP, markdown o grupo de producto.

No se imputarán valores sin evidencia.

---

### Resultado

El dataset presenta una calidad suficiente para construir una solución de **Product & Merchandising Analytics orientada a devoluciones**, siempre que se apliquen reglas explícitas de calidad y granularidad.

La auditoría también demuestra que algunas variables aparentemente simples, como `price` o `voucherAmount`, requieren entender primero su nivel de agregación antes de utilizarlas en KPIs.

### Siguiente paso

Construir una capa analítica limpia y reproducible con campos como:

- `ordered_units`
- `returned_units`
- `kept_units`
- `unit_selling_price`
- `price_status`
- `markdown_amount`
- `markdown_pct`
- `has_voucher`
- `is_zero_value_item`
- `is_valid_transaction`
- `is_valid_return`